In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [2]:
df = pd.read_csv('train_binary.csv')

In [3]:
df.describe()

,feature_0,feature_1,feature_2,feature_3,feature_4,feature_5,feature_6,feature_7,feature_8,feature_9,...,feature_16,feature_17,feature_18,feature_19,feature_20,feature_21,feature_22,feature_23,feature_24,label
count,50000.000000,50000.000000,50000.000000,50000.000000,50000.000000,50000.000000,50000.000000,50000.000000,50000.000000,50000.000000,...,50000.000000,50000.000000,50000.000000,50000.000000,50000.000000,50000.000000,50000.000000,50000.000000,50000.000000,50000.000000
mean,500.646426,0.499961,5001.475347,99.663713,0.005001,50004.809044,475.641138,0.459966,4401.601767,84.672616,...,74.912848,0.999289,2495.055679,115.182153,0.591371,2056.161274,1.277149,0.003311,-0.002680,0.303600
std,102.167451,0.010176,1018.686318,50.946605,0.001021,10193.068407,97.407907,0.009576,899.975387,44.935452,...,25.024744,0.100383,498.380114,27.473200,0.159825,418.518026,0.666971,1.042740,1.000006,0.459817
min,44.552292,0.462030,966.650179,-130.775155,0.000618,6888.787646,37.323957,0.421474,877.269883,-106.825468,...,-20.047149,0.540305,476.137094,41.911967,0.156876,729.555101,0.008627,-9.263822,-1.000000,0.000000
25%,431.558647,0.493091,4313.706744,65.277365,0.004320,43098.759523,409.719004,0.453495,3794.988088,54.217184,...,57.922540,0.932153,2156.137582,94.564638,0.471014,1686.072685,0.775074,-0.375392,-1.000000,0.000000
50%,500.723423,0.499944,5000.138426,99.838686,0.005002,50049.357109,475.743828,0.459978,4403.108013,84.700557,...,74.865170,0.999356,2497.297022,108.480117,0.546799,2181.189715,1.203037,0.000198,-1.000000,0.000000
75%,570.043673,0.506870,5686.792287,134.056839,0.005688,56864.960088,541.822674,0.466470,5008.693250,115.092937,...,91.921637,1.066220,2832.091910,136.839835,0.728151,2371.657805,1.692574,0.380763,1.000000,1.000000
max,930.857578,0.541910,9061.589352,332.284250,0.008894,95723.410815,882.181943,0.498576,8043.777043,272.166998,...,174.186524,1.461638,4572.047464,208.654969,1.167666,3058.142671,4.946865,9.609629,1.000000,1.000000


In [4]:
testdf = df.sample(frac = 0.1,random_state=42)
df.drop(testdf.index,inplace = True)

In [5]:
labels = df.label

In [6]:
df = df.drop('label',axis = 1)

In [7]:
metric = pd.DataFrame({'label' : df.columns,
              'mean' : np.mean(df,axis =0),
              'std' : np.std(df,axis = 0)})

In [8]:
metric

,label,mean,std
feature_0,feature_0,500.704484,102.302295
feature_1,feature_1,0.499940,0.010177
feature_2,feature_2,5001.771907,1019.728403
feature_3,feature_3,99.629040,50.893473
feature_4,feature_4,0.005000,0.001023
feature_5,feature_5,50020.644856,10183.702577
feature_6,feature_6,475.699537,97.535083
feature_7,feature_7,0.459947,0.009577
feature_8,feature_8,4401.869549,900.925917
feature_9,feature_9,84.622903,44.877985


In [9]:
for col in df.columns:
    if col == 'label':
        continue
    df[col] = (df[col] - np.mean(df[col]))/np.std(df[col])

In [10]:
class LogisticRegression:
    def __init__(self):
        self.optimizer = None
        self.hat_m_b = None
        self.hat_m_w = None
        self.r_v_b = None
        self.r_v_w = None
        self.hat_v_w = None
        self.hat_v_b = None
        self.x = None
        self.y = None
        self._y = None
        self.w = None
        self.b = None
        self.w_error = None
        self.b_error = None
        self.m_w = None
        self.m_b = None
        self.v_w = None
        self.v_b = None
        self.epoch_count = 1
        self.epoch_limit = None
    def predict(self,x):
        out = 1/(1 + np.exp(-(np.dot(x , self.w) + self.b)))
        return out
    def fit(self,x,y,epochs = 100,optimizer = 'gd'):
        self.x = x
        self.y = y
        self.w = np.zeros(x.shape[1])
        self.b = 0
        self._y = self.predict(self.x)
        self.epoch_limit = epochs
        self.optimizer = optimizer
        #initialized the momentum to avoid type error in first iteration
        self.v_w = np.zeros(self.x.shape[1])
        self.m_w = np.zeros(self.x.shape[1])
        self.m_b = 0
        self.v_b = 0
        #####
        self.b_error = -(np.mean(self.y - self._y))
        self.w_error = -(np.dot(self.x.transpose(),self.y - self._y))/self.x.shape[0]
        if self.optimizer == "adam":
            update_step = self.adam
        elif self.optimizer == "RMSprop":
            update_step = self.RMSprop
        elif self.optimizer == "momentum":
            update_step = self.momentum
        elif self.optimizer == "gd":
            update_step = self.gd
        else:
            raise ValueError(f"Unknown optimizer: {self.optimizer}")
        while self.epoch_count <= self.epoch_limit:
            self._y = self.predict(self.x)
            self.loss = sum([-t_hat*np.log(t) - (1-t_hat)*np.log(1-t) for t , t_hat in zip(self._y,self.y)])/len(self.y)
            self.b_error = -(np.mean(self.y - self._y))
            self.w_error = -(np.dot(self.x.transpose(),self.y - self._y))/self.x.shape[0]
            update_step()
            print(f'Epoch : {self.epoch_count} Loss : {self.loss}')
            self.epoch_count += 1
    def adam(self, beta1 = 0.9 , beta2 = 0.999 ,epsilon = 10**-8,gamma = 0.0003):
        #momentum variables
        self.m_w = beta1*self.m_w + (1 - beta1)*self.w_error
        self.m_b = beta1*self.m_b + (1 - beta1)*self.b_error
        #RMSpropFactors
        self.v_w = beta2*self.v_w + (1-beta2)*(self.w_error**2)
        self.v_b = beta2*self.v_b + (1-beta2)*(self.b_error**2)
        #bias correcting the variance values
        self.hat_v_w = self.v_w / (1 - beta2**self.epoch_count)
        self.hat_v_b = self.v_b / (1 - beta2**self.epoch_count)
        #RMSpropFactvectors for updating the final values
        self.r_v_w = 1 / (np.sqrt(self.hat_v_w + epsilon))
        self.r_v_b = 1 / (np.sqrt(self.hat_v_b + epsilon))
        #bias correcting the momentum values
        self.hat_m_w = self.m_w / (1 - beta1**self.epoch_count)
        self.hat_m_b = self.m_b / (1 - beta1**self.epoch_count)
        #Updating the final value
        self.b = self.b - self.hat_m_b*self.r_v_b*gamma
        self.w = self.w - np.multiply(self.hat_m_w,self.r_v_w)*gamma
    def RMSprop(self,beta2 = 0.999,epsilon = 10**-8, gamma = 0.001):
        self.v_w = beta2*self.v_w + (1-beta2)*(self.w_error**2)
        self.v_b = beta2*self.v_b + (1-beta2)*(self.b_error**2)
        self.r_v_w = 1 / (np.sqrt(self.v_w + epsilon))
        self.r_v_b = 1 / (np.sqrt(self.v_b + epsilon))
        self.b = self.b - self.b_error*self.r_v_b*gamma
        self.w = self.w - np.multiply(self.w_error,self.r_v_w)*gamma
    def momentum(self,beta1 = 0.9 , gamma = 0.001):
        self.m_w = beta1*self.m_w + (1 - beta1)*self.w_error
        self.m_b = beta1*self.m_b + (1 - beta1)*self.b_error
        self.b = self.b - self.m_b*gamma
        self.w = self.w - self.m_w*gamma
    def gd(self,gamma = 0.005):
        self.w = self.w - gamma*self.w_error
        self.b = self.b - self.b_error*gamma



In [12]:
model = LogisticRegression()
model.fit(df.to_numpy(),labels,optimizer='gd',epochs=1000)

Epoch : 1 Loss : 0.6931471805592402
Epoch : 2 Loss : 0.6894101154210465
Epoch : 3 Loss : 0.6857085753035498
Epoch : 4 Loss : 0.6820422014040913
Epoch : 5 Loss : 0.6784106364095968
Epoch : 6 Loss : 0.6748135245874306
Epoch : 7 Loss : 0.6712505118719375
Epoch : 8 Loss : 0.667721245947342
Epoch : 9 Loss : 0.6642253763270235
Epoch : 10 Loss : 0.6607625544292728
Epoch : 11 Loss : 0.6573324336494574
Epoch : 12 Loss : 0.6539346694289263
Epoch : 13 Loss : 0.6505689193203223
Epoch : 14 Loss : 0.6472348430496473
Epoch : 15 Loss : 0.6439321025750753
Epoch : 16 Loss : 0.6406603621424901
Epoch : 17 Loss : 0.6374192883379249
Epoch : 18 Loss : 0.6342085501368898
Epoch : 19 Loss : 0.6310278189506918
Epoch : 20 Loss : 0.6278767686698581
Epoch : 21 Loss : 0.6247550757045286
Epoch : 22 Loss : 0.6216624190223119
Epoch : 23 Loss : 0.6185984801831668
Epoch : 24 Loss : 0.6155629433717253
Epoch : 25 Loss : 0.612555495427017
Epoch : 26 Loss : 0.6095758258697602
Epoch : 27 Loss : 0.6066236269270618
Epoch : 28 L

In [13]:

tlabels = testdf.label

In [14]:
testdf.drop('label' , axis = 1, inplace = True)

In [17]:
def label_generator(x):
      z = []
      for prob in x:
        if prob > 0.5:
          z.append(1)
        else:
          z.append(0)
      return z
plabels = label_generator(model.predict(testdf.to_numpy()))

In [18]:
plabels

[1,
 0,
 0,
 0,
 0,
 1,
 0,
 0,
 1,
 1,
 0,
 0,
 1,
 0,
 0,
 0,
 1,
 0,
 0,
 1,
 0,
 0,
 0,
 1,
 0,
 0,
 0,
 0,
 0,
 1,
 1,
 1,
 0,
 1,
 0,
 0,
 0,
 0,
 1,
 0,
 0,
 0,
 1,
 0,
 1,
 1,
 1,
 0,
 0,
 1,
 1,
 0,
 1,
 1,
 0,
 1,
 1,
 1,
 0,
 1,
 1,
 0,
 0,
 0,
 1,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 1,
 0,
 0,
 1,
 0,
 1,
 1,
 0,
 0,
 0,
 1,
 0,
 0,
 0,
 0,
 0,
 1,
 1,
 1,
 1,
 0,
 1,
 1,
 0,
 0,
 0,
 0,
 1,
 0,
 1,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 1,
 0,
 0,
 0,
 0,
 1,
 0,
 1,
 0,
 1,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 1,
 1,
 0,
 0,
 0,
 0,
 0,
 1,
 0,
 0,
 0,
 0,
 0,
 1,
 0,
 0,
 0,
 0,
 1,
 0,
 1,
 0,
 0,
 0,
 0,
 0,
 1,
 0,
 0,
 1,
 0,
 1,
 0,
 0,
 0,
 0,
 1,
 0,
 0,
 1,
 0,
 1,
 0,
 1,
 0,
 0,
 0,
 1,
 0,
 1,
 0,
 0,
 0,
 1,
 0,
 0,
 1,
 0,
 1,
 0,
 0,
 1,
 0,
 0,
 0,
 1,
 0,
 0,
 0,
 1,
 0,
 0,
 0,
 1,
 0,
 0,
 1,
 1,
 0,
 1,
 0,
 0,
 0,
 0,
 0,
 1,
 1,
 1,
 0,
 0,
 0,
 1,
 1,
 1,
 1,
 0,
 1,
 0,
 0,
 1,
 1,
 0,


In [26]:
np.array(plabels) == 1

array([ True, False, False, ..., False,  True,  True], shape=(5000,))

In [27]:
def f1_score(y_true, y_pred, threshold=0.5):
    
    y_true = np.array(y_true)
    
    
    tp = np.sum((y_true == 1) & (y_pred == 1))
    fp = np.sum((y_true == 0) & (y_pred == 1))
    fn = np.sum((y_true == 1) & (y_pred == 0))
    
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    
    f1 = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0.0
    
    return f1
f1_score(np.array(tlabels),np.array(plabels))

np.float64(0.9929883138564274)

In [28]:
tdf = pd.read_csv('test_binary.csv')


In [29]:
tdf.shape

(25000, 25)

In [30]:
for col in tdf.columns:
      a = metric.loc[metric.label == col]
      std = np.float64(a.iloc[:,2].to_numpy())
      mean = np.float64(a.iloc[:,1].to_numpy())
      tdf[col] = (tdf[col] - mean)/std

C:\Users\ashis\AppData\Local\Temp\ipykernel_14140\46910231.py:3: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  std = np.float64(a.iloc[:,2].to_numpy())
C:\Users\ashis\AppData\Local\Temp\ipykernel_14140\46910231.py:4: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  mean = np.float64(a.iloc[:,1].to_numpy())


In [31]:
pred_labels = label_generator(model.predict(tdf.to_numpy()))

In [32]:
pd.value_counts(pred_labels)

C:\Users\ashis\AppData\Local\Temp\ipykernel_14140\3154038371.py:1: FutureWarning: pandas.value_counts is deprecated and will be removed in a future version. Use pd.Series(obj).value_counts() instead.
  pd.value_counts(pred_labels)
C:\Users\ashis\AppData\Local\Temp\ipykernel_14140\3154038371.py:1: FutureWarning: value_counts with argument that is not not a Series, Index, ExtensionArray, or np.ndarray is deprecated and will raise in a future version.
  pd.value_counts(pred_labels)


1    21963
0     3037
Name: count, dtype: int64

In [33]:
preddf = pd.DataFrame({'prediction' : pred_labels})
preddf.to_csv('val_binary.csv')